# SAE Feature Intervention on Qwen3.6-35B-A3B L17

**Hypothesis**: logreg direction patching gave null causality (+5pp, sub-threshold). SAE features (polysemantic-free) may give stronger causal control, per arXiv:2510.07364 ("Base Models Know How to Reason, Thinking Models Learn When") which recovered 76-91% of base→thinking gap via SAE steering on Qwen2.5-32B dense.

**This notebook**: first SAE feature-level intervention on Qwen3.6-35B-A3B hybrid MoE+GDN+Gated-Attn.

**Pipeline** (~4h total on RTX 6000 Blackwell 96GB):
1. Load model + Stage B L17 residuals (~5 min)
2. Train TopK SAE on L17 residuals (d_sae=8192, k=32, ~1.5h)
3. ReasonScore: find per-letter top-20 features (~5 min)
4. Feature intervention — patch feature activation at T=10 (N=20 rollouts, 3 configs, ~2h)
5. Analysis: SAE-feature effect vs logreg-direction baseline (our V1/V2 results)

**Two decisive outcomes**:
- SAE flip_rate > 20% → **first causal evidence in hybrid MoE** → paper seção 3 ganha
- SAE flip_rate ≈ logreg (~5%) → **stronger negative result** → planning em hybrid precisa de multi-layer circuit, não single-feature

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    pip('install','-q','--no-cache-dir',
        'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/'
        'causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl',
        check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping colab auth: {e})')

import json, re, pickle, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/sae_intervention')
OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load model (needed for intervention, NOT for SAE training)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
PATCH_LAYER = 17

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Extract L17 per-token residuals from Stage B (training data for SAE)

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))
print(f'{len(shards)} shards')

# Collect ALL per-token residuals at L17 for SAE training.
# Also collect rollout_info + per-rollout token-position activations for intervention.
MIN_LEN = 200
all_tokens = []  # [(N_total_tokens, d_model)] — flat
rollout_info = []
rollout_acts_by_idx = {}  # rollout_idx -> tensor [response_len, d_model]
response_tokens_all = []

rollout_idx_global = 0
for shard in tqdm(shards, desc='collect L17'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])[f'L{PATCH_LAYER}']
        rollouts = json.loads(meta['rollouts'])
        rts = json.loads(meta['response_tokens'])
        q_options = json.loads(meta['options'])
        q_idx = int(meta['question_global_idx'])
        acts = f.get_tensor(f'acts_L{PATCH_LAYER}')  # [total_tok, d_model] bfloat16
        for r_idx, r in enumerate(rollouts):
            if r['pred'] is None or r['response_len'] < MIN_LEN:
                continue
            rollout_acts = acts[offs[r_idx]:offs[r_idx+1]].clone()  # [T, d_model]
            all_tokens.append(rollout_acts)
            rollout_acts_by_idx[rollout_idx_global] = rollout_acts
            rollout_info.append(dict(
                question=meta['question'], options=q_options, gold=meta['gold_letter'],
                pred=r['pred'], correct=r['correct'], question_idx=q_idx,
                rollout_id=r['rollout_id'], idx_global=rollout_idx_global))
            response_tokens_all.append(rts[r_idx])
            rollout_idx_global += 1

X_train = torch.cat(all_tokens, dim=0).float()  # [N_total_tokens, d_model]
d_model = X_train.shape[1]
letters = sorted(set(r['pred'] for r in rollout_info))
print(f'Training tokens: {X_train.shape[0]:,} | d_model: {d_model}')
print(f'Rollouts: {len(rollout_info)} | letters: {letters}')

# Save rollout metadata for reuse
with open(OUT/'rollout_meta.pkl', 'wb') as f:
    pickle.dump(dict(rollout_info=rollout_info, response_tokens_all=response_tokens_all,
                     letters=letters, d_model=d_model), f)

## Cell 4 — Train TopK SAE on L17 residuals

TopK activation (k=32 active features of 8192) — follows Gao et al. 2024 recipe.
Non-linear loss: MSE reconstruction + auxiliary dead-feature loss.

Config: d_sae = 4× d_model = 8192, k=32, lr=3e-4, batch=1024, ~50k steps
Wall: ~1-1.5h on RTX 6000 Blackwell

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model = d_model
        self.d_sae = d_sae
        self.k = k
        self.W_enc = nn.Parameter(torch.randn(d_model, d_sae) * (1.0/d_model**0.5))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_model) * (1.0/d_sae**0.5))
        self.b_dec = nn.Parameter(torch.zeros(d_model))
        # Normalize decoder rows to unit norm
        with torch.no_grad():
            self.W_dec.data = self.W_dec.data / self.W_dec.data.norm(dim=-1, keepdim=True)

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        # TopK: keep top-k activations per token, rest = 0
        topk_vals, topk_idx = pre.topk(self.k, dim=-1)
        z = torch.zeros_like(pre)
        z.scatter_(-1, topk_idx, F.relu(topk_vals))
        return z

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        return x_hat, z

# Config
D_SAE = 8192
K_ACTIVE = 32
LR = 3e-4
BATCH = 1024
STEPS = 30_000

sae = TopKSAE(d_model, D_SAE, K_ACTIVE).cuda()
opt = torch.optim.Adam(sae.parameters(), lr=LR, betas=(0.9, 0.999))

# Move training data to GPU
X_gpu = X_train.cuda()
N = X_gpu.shape[0]
print(f'Training on {N:,} tokens')

# Initialize b_dec as mean of data
with torch.no_grad():
    sae.b_dec.data = X_gpu.mean(dim=0)

losses = []
feat_activated = torch.zeros(D_SAE, device='cuda')  # track dead features

sae.train()
t0 = time.time()
for step in tqdm(range(STEPS), desc='SAE train'):
    idx = torch.randint(0, N, (BATCH,), device='cuda')
    x = X_gpu[idx]
    x_hat, z = sae(x)
    mse = F.mse_loss(x_hat, x)
    loss = mse
    opt.zero_grad()
    loss.backward()
    # Unit-norm constraint on decoder via gradient projection
    with torch.no_grad():
        grad = sae.W_dec.grad
        proj = (grad * sae.W_dec.data).sum(dim=-1, keepdim=True) * sae.W_dec.data
        sae.W_dec.grad = grad - proj
    opt.step()
    with torch.no_grad():
        sae.W_dec.data = sae.W_dec.data / sae.W_dec.data.norm(dim=-1, keepdim=True).clamp(min=1e-8)
        feat_activated += (z > 0).float().sum(dim=0)
    losses.append(loss.item())
    if (step+1) % 2000 == 0:
        recent = np.mean(losses[-500:])
        n_dead = (feat_activated == 0).sum().item()
        print(f'  step {step+1:5d} | mse {recent:.4f} | dead features {n_dead}/{D_SAE}')
        feat_activated.zero_()  # reset rolling counter

wall_min = (time.time() - t0) / 60
print(f'\nSAE training done in {wall_min:.1f} min, final mse: {np.mean(losses[-500:]):.4f}')

# Normalized reconstruction: check fidelity
sae.eval()
with torch.no_grad():
    # Random 10k sample
    s_idx = torch.randint(0, N, (10_000,), device='cuda')
    x_s = X_gpu[s_idx]
    x_hat, z = sae(x_s)
    recon_cos = F.cosine_similarity(x_hat, x_s, dim=-1).mean().item()
    recon_var = 1.0 - ((x_hat - x_s).var(dim=0) / x_s.var(dim=0)).mean().item()
    l0 = (z > 0).float().sum(dim=-1).mean().item()
print(f'Recon cos_sim: {recon_cos:.4f} | variance explained: {recon_var:.4f} | L0: {l0:.1f}')

torch.save(sae.state_dict(), OUT/'sae_L17.pt')
print(f'saved SAE to {OUT/"sae_L17.pt"}')

## Cell 5 — ReasonScore: find per-letter top features

For each letter L ∈ {A..J}:
- For each feature f: compute mean activation on rollouts predicting L vs not-L
- Rank features by `(mean_L - mean_not_L) / std` — effect size
- Take top-20 features per letter

In [ ]:
# Encode all per-rollout first-10-tokens activations into SAE features
sae.eval()
features_per_rollout = {}  # rollout_idx -> [T, d_sae] SAE activations
with torch.no_grad():
    for idx, acts in tqdm(rollout_acts_by_idx.items(), desc='encode SAE'):
        x = acts[:10].float().cuda()  # first 10 tokens (silent plan window)
        z = sae.encode(x)
        features_per_rollout[idx] = z.cpu().numpy()  # [T=10, d_sae]

# Per-letter top features via effect size
# Pool first-10 activations per rollout (mean), then compare by letter
pooled_per_rollout = np.stack([features_per_rollout[r['idx_global']].mean(axis=0) for r in rollout_info])
# pooled_per_rollout: [N_rollouts, d_sae]

labels_per_rollout = np.array([r['pred'] for r in rollout_info])
top_features_per_letter = {}
for letter in letters:
    mask = (labels_per_rollout == letter)
    mean_in = pooled_per_rollout[mask].mean(axis=0)
    mean_out = pooled_per_rollout[~mask].mean(axis=0)
    std_all = pooled_per_rollout.std(axis=0) + 1e-6
    effect = (mean_in - mean_out) / std_all  # [d_sae]
    top20 = np.argsort(-effect)[:20]
    top_features_per_letter[letter] = dict(
        features=top20.tolist(),
        effects=effect[top20].tolist(),
        mean_activation=mean_in[top20].tolist())
    print(f'{letter}: top5 features {top20[:5].tolist()} effects {effect[top20[:5]].round(3)}')

with open(OUT/'top_features.json', 'w') as f:
    json.dump(top_features_per_letter, f, indent=2)
print('\n✅ top features per letter saved')

## Cell 6 — Hook + intervention function

Intervention: at T=10 position, ADD `α × W_dec[target_top_features]` to residual, SUBTRACT `α × W_dec[source_top_features]`.

This is the minimal SAE-steering analog of logreg-direction patch, but in feature space.

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

# Precompute per-letter intervention vectors in residual space
# v_letter = sum over top-20 features of W_dec[f] weighted by activation target
# We'll use uniform weighting (1.0 per feature) — or weight by effect size — tunable via ALPHA
W_dec = sae.W_dec.data.detach().cpu().numpy()  # [d_sae, d_model]

feature_vec_per_letter = {}
for letter in letters:
    top_f = top_features_per_letter[letter]['features'][:20]
    weights = np.array(top_features_per_letter[letter]['effects'][:20], dtype=np.float32)
    weights = weights / np.linalg.norm(weights)
    # Weighted sum of decoder vectors (this IS the letter direction in feature space)
    v = (W_dec[top_f] * weights[:, None]).sum(axis=0)
    v = v / (np.linalg.norm(v) + 1e-8)
    feature_vec_per_letter[letter] = v.astype(np.float32)

print('✅ per-letter feature vectors (unit norm):')
for l in letters: print(f'  {l}: shape {feature_vec_per_letter[l].shape}, norm {np.linalg.norm(feature_vec_per_letter[l]):.4f}')

class PatchController:
    def __init__(self):
        self.patch_vector = None
        self.prompt_len = None
        self.target_response_pos = None
        self.active = False
        self.applied = False

    def start(self, patch_vec, prompt_len, target_response_pos):
        self.patch_vector = torch.from_numpy(patch_vec).to('cuda', torch.bfloat16)
        self.prompt_len = prompt_len
        self.target_response_pos = target_response_pos
        self.active = True
        self.applied = False

    def stop(self):
        self.active = False
        self.patch_vector = None

    def make_hook(self):
        def hook(module, inp, out):
            if not self.active or self.applied: return out
            hidden = out[0] if isinstance(out, tuple) else out
            target_abs_pos = self.prompt_len + self.target_response_pos
            if hidden.shape[1] > target_abs_pos:
                hidden = hidden.clone()
                hidden[:, target_abs_pos, :] += self.patch_vector
                self.applied = True
                if isinstance(out, tuple):
                    return (hidden,) + out[1:]
                return hidden
            return out
        return hook

controller = PatchController()
layer_module = get_layer_module(model, PATCH_LAYER)
hook_handle = layer_module.register_forward_hook(controller.make_hook())
print(f'hook installed on L{PATCH_LAYER}')

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
def extract_letter(c, n_opts):
    m = list(BOXED_RE.finditer(c))
    if m:
        l = m[-1].group(1).upper()
        if ord(l)-ord('A') < n_opts: return l
    tail = c[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands and ord(cands[-1])-ord('A') < n_opts: return cands[-1]
    return None

def format_prompt(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason step by step, "
        "then put the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

def generate_with_patch(prompt, forced_response_prefix_tokens, patch_vec, max_new=1024):
    enc = tok(prompt, return_tensors='pt').to('cuda')
    prompt_len = enc.input_ids.shape[1]
    forced_ids = torch.tensor([forced_response_prefix_tokens], device='cuda')
    full_input = torch.cat([enc.input_ids, forced_ids], dim=1)
    full_attn = torch.ones_like(full_input)
    patch_pos = len(forced_response_prefix_tokens) - 1
    controller.stop()
    if patch_vec is not None:
        controller.start(patch_vec, prompt_len, patch_pos)
    try:
        with torch.no_grad():
            out = model.generate(
                input_ids=full_input, attention_mask=full_attn,
                max_new_tokens=max_new, do_sample=False,
                pad_token_id=tok.pad_token_id, use_cache=True,
            )
    finally:
        controller.stop()
    generated = out[0, full_input.shape[1]:]
    full_response = torch.cat([forced_ids[0], generated])
    return tok.decode(full_response, skip_special_tokens=True)

print('✅ intervention function ready')

## Cell 7 — Run SAE-feature intervention (N=20, ~45 min)

Per rollout, 3 generations:
1. **baseline** — no patch
2. **patch** — add α·(v_target - v_source) at T=10, L17
3. **null** — random direction same norm

Compare with V1 (logreg L11 α=5, flip=5% null=5%, effect=0pp) and V2 (logreg L17 α=12, flip=5% null=0%, effect=+5pp).

In [ ]:
correct_mask = np.array([r['correct'] for r in rollout_info], dtype=bool)
confident_idx = np.where(correct_mask)[0]
np.random.seed(42)
np.random.shuffle(confident_idx)

N_INTERVENTIONS = 20
PATCH_ALPHA = 12.0  # same as V2 for apples-to-apples
PATCH_POSITION = 10

results = []
t0 = time.time()
for i, rollout_idx_int in enumerate(tqdm(confident_idx[:N_INTERVENTIONS], desc='sae-intervene')):
    rollout_idx = int(rollout_idx_int)
    info = rollout_info[rollout_idx]
    source_letter = info['pred']
    letters_avail = [l for l in feature_vec_per_letter if l != source_letter and ord(l)-ord('A') < len(info['options'])]
    if not letters_avail: continue
    target_letter = random.Random(rollout_idx).choice(letters_avail)

    prompt = format_prompt(info['question'], info['options'])
    forced_prefix = response_tokens_all[rollout_idx][:PATCH_POSITION]

    v_source = feature_vec_per_letter[source_letter]
    v_target = feature_vec_per_letter[target_letter]
    patch_vec = PATCH_ALPHA * (v_target - v_source)

    text_baseline = generate_with_patch(prompt, forced_prefix, None)
    letter_baseline = extract_letter(text_baseline, len(info['options']))

    text_patched = generate_with_patch(prompt, forced_prefix, patch_vec)
    letter_patched = extract_letter(text_patched, len(info['options']))

    rng = np.random.RandomState(rollout_idx * 31 + 7)
    rand_dir = rng.randn(len(v_source)).astype(np.float32)
    rand_dir = rand_dir / np.linalg.norm(rand_dir) * PATCH_ALPHA
    text_random = generate_with_patch(prompt, forced_prefix, rand_dir)
    letter_random = extract_letter(text_random, len(info['options']))

    results.append(dict(
        idx=rollout_idx, source=source_letter, target=target_letter,
        baseline=letter_baseline, patched=letter_patched, random=letter_random,
        gold=info['gold'], n_options=len(info['options']),
    ))
    if (i+1) % 4 == 0:
        n_flip = sum(1 for r in results if r['patched'] == r['target'])
        n_keep = sum(1 for r in results if r['patched'] == r['source'])
        n_null = sum(1 for r in results if r['random'] == r['target'])
        print(f'  [{i+1}/{N_INTERVENTIONS}] SAE flip: {n_flip}/{len(results)} | kept source: {n_keep} | null flip: {n_null}')

with open(OUT/'sae_intervention_results.json', 'w') as f:
    json.dump(dict(layer=PATCH_LAYER, alpha=PATCH_ALPHA, pos=PATCH_POSITION,
                   d_sae=D_SAE, k=K_ACTIVE, top_k_features=20,
                   n=len(results), results=results,
                   wall_min=(time.time()-t0)/60), f, indent=2)
print(f'\n✅ done in {(time.time()-t0)/60:.1f} min')

## Cell 8 — Analysis + comparison vs logreg V1/V2

In [ ]:
from collections import Counter

def tally(results, key):
    c = Counter()
    for r in results:
        if r[key] is None: c['invalid'] += 1
        elif r[key] == r['source']: c['source'] += 1
        elif r[key] == r['target']: c['target'] += 1
        else: c['other'] += 1
    return c

print('=== SAE-feature intervention (L17, α=12, top-20 features) ===\n')
for key in ['baseline', 'patched', 'random']:
    c = tally(results, key)
    total = sum(c.values())
    print(f'{key:10s}: ' + ' '.join(f'{k}={v}/{total}({v/total*100:.0f}%)' for k,v in c.items()))

flip_sae = tally(results, 'patched')['target'] / len(results)
null_sae = tally(results, 'random')['target'] / len(results)
effect_sae = flip_sae - null_sae
print(f'\nSAE effect: flip={flip_sae*100:.1f}% null={null_sae*100:.1f}% → effect={effect_sae*100:+.1f}pp')

# Comparison table
print('\n=== Comparison: logreg vs SAE ===')
comparison = [
    ('logreg V1', 'L11', 5.0,  10, 5.0, 5.0),
    ('logreg V2', 'L17', 12.0, 15, 5.0, 0.0),
    ('SAE',      'L17', 12.0, 10, flip_sae*100, null_sae*100),
]
print(f'{"method":12s} {"layer":6s} {"alpha":>6s} {"pos":>4s} {"flip%":>6s} {"null%":>6s} {"effect":>8s}')
for m, l, a, p, f, n in comparison:
    print(f'{m:12s} {l:6s} {a:>6.1f} {p:>4d} {f:>5.1f}% {n:>5.1f}% {f-n:>+7.1f}pp')

# Verdict
print('\n=== VERDICT ===')
if effect_sae > 0.30:
    print('⭐⭐⭐ STRONG CAUSAL — SAE features causally control reasoning in hybrid MoE')
    print('   → First such evidence in Qwen3.6 triple-hybrid arch. Paper section 3 → strong.')
elif effect_sae > 0.15:
    print('⭐⭐ MODERATE CAUSAL — SAE features move the needle but not reliably')
    print('   → Paper section 3 → moderate with discussion of boundaries.')
elif effect_sae > 0.05:
    print('⭐ WEAK CAUSAL — SAE better than logreg but still sub-threshold')
    print('   → Single-feature intervention insufficient. Multi-feature circuits next.')
else:
    print('❌ NULL — SAE features also do not flip in hybrid MoE')
    print('   → STRONGER negative result: even polysemantic-free features insufficient.')
    print('   → Motivates transcoders + attribution graphs (multi-layer circuits) as next.')
print(f'\nsaved to {OUT/"sae_intervention_results.json"}')